# 38 CPU audio fusion compare

audio raw/enhanced/separated/embedding を player-season projection、audio-aware late fusion、method evaluation に接続します。

Creates: audio run の player-season projection、`predictions/{fusion_audio_run_id}/`、`reports/method_evaluation/{method_evaluation_audio_report_id}/`。既存 v2 report は上書きしません。

Required inputs: audio branch の `predictions_v1.parquet` と既存 v2 context/sequence/video/VLM/fusion 比較用 predictions。

Next: HTML report を見て、同一 sample intersection と既存 fusion / audio-aware fusion の差を見る。

In [ ]:
from pathlib import Path
import json
import os
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or already mounted:', exc)

os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff_with_audio')
repo_dir = Path(os.environ['BATTING_CODE_ROOT'])
if not (repo_dir / 'src' / 'sport_pipeline').exists():
    repo_dir = Path.cwd()
sys.path.insert(0, str(repo_dir / 'src'))

profile = json.loads((repo_dir / 'configs' / 'runs' / os.environ['BASEBALL_VISION_RUN_PROFILE']).read_text(encoding='utf-8'))
BASE_DIR = Path(profile['paths']['base_dir'])
RUN_IDS = profile['run_ids']
FUSION_EXEC = profile['execution']['fusion']
audio_runs = [RUN_IDS[key] for key in ['audio_raw_run_id', 'audio_enhanced_run_id', 'audio_separated_run_id', 'audio_embedding_run_id']]
print('audio runs:', audio_runs)

In [ ]:
for run_id in audio_runs:
    path = BASE_DIR / 'predictions' / run_id / 'predictions_v1.parquet'
    print(('OK      ' if path.exists() else 'MISSING '), path)

In [ ]:
from sport_pipeline.models.player_season.event_projection import run_event_prediction_player_season_projection

projection_outputs = {}
for source_run_id in audio_runs:
    projection_outputs[source_run_id] = run_event_prediction_player_season_projection(
        BASE_DIR,
        source_run_id=source_run_id,
        require_non_empty=False,
    )
print(json.dumps({k: {kk: str(vv) for kk, vv in v.items()} for k, v in projection_outputs.items()}, indent=2, ensure_ascii=False))

In [ ]:
from sport_pipeline.models.fusion.run_fusion import run_full_fusion

source_runs = FUSION_EXEC.get('source_runs') or []
fusion_outputs = run_full_fusion(
    BASE_DIR,
    fusion_run_id=RUN_IDS['fusion_audio_run_id'],
    source_runs=source_runs,
    learn_weights_from_validation=FUSION_EXEC.get('learn_weights_from_validation', False),
)
print(json.dumps({k: str(v) for k, v in fusion_outputs.items()}, indent=2, ensure_ascii=False))

In [ ]:
from sport_pipeline.reports.method_evaluation import build_method_evaluation_report

audio_report_id = RUN_IDS.get('method_evaluation_audio_report_id', 'method_evaluation_with_audio_mlb_2024_2026_v2')
report_outputs = build_method_evaluation_report(BASE_DIR, profile, report_id=audio_report_id, include_fusion_reference=True)
print(json.dumps({k: str(v) for k, v in report_outputs.items()}, indent=2, ensure_ascii=False))
print('Open:', report_outputs['html'])